# China PIPE / 定向增发 — Data Load Demo

This notebook demonstrates scraping private placement (PIPE) data from East Money and computing placement discounts via baostock.

| Field | Description |
|---|---|
| `announcement_date` | Date shares were issued / priced |
| `listing_date` | Date new shares started trading |
| `lock_length_months` | Contractual lock-up period (months) |
| `issue_price` | Price per share placed (CNY) |
| `reference_price` | 20-trading-day avg close before issue date (CNY) |
| `discount_pct` | `(reference − issue) / reference × 100` |
| `shares_issued` | Number of shares placed |
| `amount_raised_cny` | Gross proceeds (CNY) |

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Fetch raw PIPE records (no reference prices)

We call `EastMoneyPIPEScraper` directly so we can filter to 2010–2011 before running the slower baostock reference-price step.

In [ ]:
from scraper import EastMoneyPIPEScraper, ReferencePriceCalculator, SCHEMA

# fetch all 定向增发 records (scraper filters to 2010-2024 by default)
raw = EastMoneyPIPEScraper().fetch()
print(f'Total records (2010-2024): {len(raw):,}')

In [ ]:
# narrow to 2010-2011
mask = (
    raw['announcement_date'].between('2010-01-01', '2011-12-31')
    | raw['listing_date'].between('2010-01-01', '2011-12-31')
)
df = raw[mask].copy().reset_index(drop=True)
print(f'Records in 2010-2011: {len(df):,}')
df.head()

## 2. Compute reference prices & discounts via baostock

> **Note**: this step makes one baostock API call per unique `(stock_code, announcement_date)` pair.
> For ~174 records it takes roughly **2–4 minutes**. Skip to §3 to load cached data instead.

In [ ]:
df = ReferencePriceCalculator(lookback_days=20).compute(df)
print(f'Reference price coverage: {df["reference_price"].notna().sum()} / {len(df)}')
print(f'Discount coverage:        {df["discount_pct"].notna().sum()} / {len(df)}')

## 3. Save & reload (skip baostock on re-runs)

In [ ]:
OUT = os.path.join('data', 'china_pipes_2010_2011.csv')
os.makedirs('data', exist_ok=True)
df.to_csv(OUT, index=False, encoding='utf-8-sig')
print(f'Saved → {OUT}')

In [ ]:
# reload from cache on subsequent runs
# df = pd.read_csv(OUT, parse_dates=['announcement_date', 'listing_date'])
# print(f'Loaded {len(df):,} records')

## 4. Quick data quality check

In [ ]:
print('── Schema ──────────────────────────────')
print(df.dtypes)
print()
print('── Null counts ─────────────────────────')
print(df.isna().sum())
print()
print('── Date range ──────────────────────────')
print(f'  announcement_date: {df["announcement_date"].min()}  →  {df["announcement_date"].max()}')

In [ ]:
df[['stock_code', 'company_name', 'announcement_date',
    'lock_length_months', 'issue_price', 'reference_price',
    'discount_pct', 'amount_raised_cny']].describe()

## 5. Lock-up period distribution

In [ ]:
lock_counts = (
    df['lock_length_months']
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(8, 4))
lock_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Lock-up Period Distribution — China PIPEs 2010–2011')
ax.set_xlabel('Lock-up (months)')
ax.set_ylabel('Number of placements')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print(lock_counts.rename('count').to_frame())

## 6. Discount distribution

In [ ]:
disc = df['discount_pct'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(disc, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(disc.median(), color='firebrick', linestyle='--',
                label=f'Median {disc.median():.1f}%')
axes[0].set_title('Discount Distribution')
axes[0].set_xlabel('Discount (%)')
axes[0].set_ylabel('Count')
axes[0].legend()

sorted_disc = disc.sort_values()
cdf = (sorted_disc.rank(method='max') / len(sorted_disc)) * 100
axes[1].plot(sorted_disc.values, cdf.values, color='steelblue')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Cumulative Distribution of Discount')
axes[1].set_xlabel('Discount (%)')
axes[1].set_ylabel('Cumulative %')
axes[1].grid(True, alpha=0.3)

plt.suptitle('China PIPEs 2010–2011', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Discount summary (%):')
print(disc.describe().to_frame().T.to_string(index=False))

## 7. Timeline — placements per month

In [ ]:
monthly = (
    df.assign(announcement_date=pd.to_datetime(df['announcement_date']))
    .set_index('announcement_date')
    .resample('MS')['stock_code']
    .count()
)

fig, ax = plt.subplots(figsize=(12, 4))
monthly.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('China PIPEs per Month — 2010–2011')
ax.set_xlabel('')
ax.set_ylabel('Number of placements')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 8. Sample records

In [ ]:
df[['stock_code', 'company_name', 'announcement_date', 'listing_date',
    'lock_length_months', 'issue_price', 'reference_price',
    'discount_pct', 'shares_issued', 'amount_raised_cny']].head(15)